## OS-Level Action을 사용하는 AgentCore Browser Tool(boto3 SDK)

이 Notebook에서는 **boto3 SDK**를 사용해 [Amazon Bedrock AgentCore Browser](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore-browser.html)에서 **OS-level action**(`InvokeBrowser`)을 실행하는 방법을 보여 줍니다. SigV4를 수동으로 서명할 필요가 없습니다.

OS-level action을 사용하면 CDP/Playwright 자동화 계층을 완전히 우회해 브라우저 sandbox에서 mouse, keyboard, screenshot 및 scroll 작업을 직접 수행할 수 있습니다. 다음 대상과 상호 작용할 때 유용합니다.

- **OS-native dialog** - File upload/download prompt, print dialog, 인증 pop-up
- **Browser chrome element** - Address bar, extension popup, 권한 banner
- **Keyboard shortcut** - CDP 기반 자동화에서 OS로 전송할 수 없는 `Ctrl+S`, `Ctrl+A`, `Alt+Tab`
- **Canvas / WebGL content** - DOM selector가 없는 content
- 기존 CDP 기반 자동화로 처리하기 어려운 **모든 element**

#### 수행할 작업

1. 사용자 지정 AgentCore Browser 생성(control plane)
2. 브라우저 세션 시작(data plane)
3. `invoke_browser()`로 OS-level action 실행: Mouse(click, move, drag), scroll, keyboard(type, press, shortcut), screenshot
4. 리소스 정리

#### 사전 요구 사항

- `boto3 >= 1.42.85`(`invoke_browser` 지원 포함)
- `bedrock-agentcore` 권한이 있는 AWS 자격 증명

### 1. Dependency 설치

In [ ]:
!uv pip install -qU -r requirements.txt

import boto3

print(f"boto3: {boto3.__version__}")

### 2. AWS 자격 증명 불러오기

실습을 위해 terminal에서 자격 증명 file을 생성합니다.

In [ ]:
import os
from pathlib import Path

env_file = Path(".env")
assert env_file.exists(), "Missing .env — run: ./setup_aws_creds.sh <account_id>"

for line in env_file.read_text().splitlines():
    if "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip()

assert os.environ.get("AWS_ACCESS_KEY_ID"), "Credentials not loaded"
print("AWS credentials loaded ✓")

### 3. boto3 client 설정

In [ ]:
import time

REGION = "us-west-2"
BROWSER_NAME = "browser_with_os_actions"

# Control plane: 브라우저 생성/삭제
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

# Data plane: 세션 및 invoke
runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"Clients ready — region: {REGION}")

### 3.1 IAM Role 생성

In [ ]:
import sys

sys.path.insert(0, str(Path.cwd()))

from helpers.utils import create_agentcore_execution_role, SAMPLE_ROLE_NAME

execution_role_arn = create_agentcore_execution_role(SAMPLE_ROLE_NAME)

# IAM 전파 대기
print("Waiting 10s for IAM propagation...")
time.sleep(10)

### 4. AgentCore Browser 생성

In [ ]:
created = control_client.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=execution_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
)

browser_id = created["browserId"]
print(f"Browser created: {browser_id}")

### 5. 브라우저 세션 시작

In [ ]:
session = runtime_client.start_browser_session(
    browserIdentifier=browser_id,
    name=BROWSER_NAME,
    sessionTimeoutSeconds=3600,
    viewPort={"width": 1920, "height": 1080},
)

session_id = session["sessionId"]
print(f"Session started: {session_id}")

print("Waiting 3s for session init...")
time.sleep(3)

### Helper - invoke wrapper 사용

Action 셀을 간결하게 유지하기 위한 작은 wrapper입니다.

In [ ]:
def invoke(action: dict) -> dict:
    """브라우저 작업을 호출하고 결과 딕셔너리를 반환합니다."""
    resp = runtime_client.invoke_browser(browserIdentifier=browser_id, sessionId=session_id, action=action)
    return resp["result"]


print("invoke() helper ready")

### 6. Mouse Action 실행

In [ ]:
print("── Mouse Click Actions ──\n")

# 왼쪽 click
r = invoke({"mouseClick": {"x": 600, "y": 370, "button": "LEFT"}})
print(f"Left click: {r}")

# 두 번 click
r = invoke({"mouseClick": {"x": 500, "y": 300, "button": "LEFT", "clickCount": 2}})
print(f"Double click: {r}")

# 오른쪽 click
r = invoke({"mouseClick": {"x": 200, "y": 400, "button": "RIGHT", "clickCount": 1}})
print(f"Right click: {r}")

# 가운데 click
r = invoke({"mouseClick": {"x": 960, "y": 540, "button": "MIDDLE", "clickCount": 1}})
print(f"Middle click: {r}")

# 여러 번 click
r = invoke({"mouseClick": {"x": 500, "y": 300, "button": "LEFT", "clickCount": 10}})
print(f"Multi click (10x): {r}")

In [ ]:
print("── Mouse Move Actions ──\n")

r = invoke({"mouseMove": {"x": 800, "y": 600}})
print(f"Move to (800,600): {r}")

r = invoke({"mouseMove": {"x": 1, "y": 1}})
print(f"Move to (1,1): {r}")

In [ ]:
print("── Mouse Drag Actions ──\n")

r = invoke(
    {
        "mouseDrag": {
            "startX": 1,
            "startY": 1,
            "endX": 705,
            "endY": 180,
            "button": "LEFT",
        }
    }
)
print(f"Drag (1,1)->(705,180): {r}")

r = invoke(
    {
        "mouseDrag": {
            "startX": 1,
            "startY": 1,
            "endX": 370,
            "endY": 330,
            "button": "LEFT",
        }
    }
)
print(f"Drag (1,1)->(370,330): {r}")

r = invoke(
    {
        "mouseDrag": {
            "startX": 500,
            "startY": 300,
            "endX": 100,
            "endY": 200,
            "button": "MIDDLE",
        }
    }
)
print(f"Drag middle button: {r}")

### 7. Scroll Action 실행

In [ ]:
print("── Scroll Actions ──\n")

r = invoke({"mouseScroll": {"x": 800, "y": 600, "deltaX": 0, "deltaY": -500}})
print(f"Scroll up: {r}")

r = invoke({"mouseScroll": {"x": 500, "y": 300, "deltaX": 300, "deltaY": 0}})
print(f"Scroll right: {r}")

r = invoke({"mouseScroll": {"x": 500, "y": 300, "deltaX": -100, "deltaY": -200}})
print(f"Scroll diagonal: {r}")

r = invoke({"mouseScroll": {"x": 500, "y": 300, "deltaX": 1000, "deltaY": 1000}})
print(f"Scroll large: {r}")

### 8. Keyboard Action 실행

In [ ]:
print("── Key Type Actions ──\n")

r = invoke({"keyType": {"text": "Hello World"}})
print(f"Type 'Hello World': {r}")

r = invoke({"keyType": {"text": "user@example.com!#$%^&*()"}})
print(f"Type special chars: {r}")

r = invoke({"keyType": {"text": "https://www.example.com"}})
print(f"Type URL: {r}")

r = invoke({"keyType": {"text": "1" * 10000}})
print(f"Type long string (10k): {r}")

In [ ]:
print("── Key Press Actions ──\n")

r = invoke({"keyPress": {"key": "enter"}})
print(f"Enter: {r}")

r = invoke({"keyPress": {"key": "tab"}})
print(f"Tab: {r}")

r = invoke({"keyPress": {"key": "escape"}})
print(f"Escape: {r}")

r = invoke({"keyPress": {"key": "backspace", "presses": 5}})
print(f"Backspace x5: {r}")

r = invoke({"keyPress": {"key": "ArrowDown", "presses": 100}})
print(f"ArrowDown x100: {r}")

In [ ]:
print("── Key Shortcut Actions ──\n")

r = invoke({"keyShortcut": {"keys": ["ctrl", "s"]}})
print(f"Ctrl+S: {r}")

r = invoke({"keyShortcut": {"keys": ["ctrl", "p"]}})
print(f"Ctrl+P: {r}")

r = invoke({"keyShortcut": {"keys": ["ctrl", "shift", "i"]}})
print(f"Ctrl+Shift+I: {r}")

### 9. Screenshot 확인

In [ ]:
from IPython.display import Image, display
import base64


def take_screenshot(fmt="PNG"):
    """스크린샷을 촬영해 인라인으로 표시합니다."""
    action = {"screenshot": {"format": fmt}} if fmt else {"screenshot": {}}
    result = invoke(action)
    print(f"Screenshot status: {result.get('screenshot', {})}")

    data = result.get("screenshot", {}).get("data")
    if data:
        # 데이터는 boto3에서 bytes(blob type)로 전달됨
        img_bytes = data if isinstance(data, bytes) else base64.b64decode(data)
        display(Image(img_bytes))
    else:
        print("No screenshot data returned")


print("── Screenshot (PNG) ──")
take_screenshot("PNG")

print("\n── Screenshot (default format) ──")
take_screenshot(None)

### 10. 브라우저 세션 중지

In [ ]:
runtime_client.stop_browser_session(browserIdentifier=browser_id, sessionId=session_id)
print(f"Session {session_id} stopped ✓")

### 11. 정리(선택 사항)

브라우저와 IAM role을 삭제합니다.

In [ ]:
control_client.delete_browser(browserId=browser_id)
print(f"Browser {browser_id} deleted ✓")

In [ ]:
from helpers.utils import delete_agentcore_execution_role, SAMPLE_ROLE_NAME

delete_agentcore_execution_role(SAMPLE_ROLE_NAME)
print("IAM role cleaned up ✓")